1. 核心零件:
- 多头注意力    Multi-Head Attention
- 掩码(防作弊)  Masking
- 前馈网络(消化吸收)    Feed forward
- 归一化与残差(稳定训练)    LayerNorm & Residuals

In [ ]:
# 把2.ipynb的核心框架拉过来拼在一起
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib

class Block(nn.Module):
    '''一个完整的Transformer块'''
    def __init__(self, n_embd, n_head):
        '''
        n_embd: 词嵌入维度(Embedding Dimension)：每个词被转化成的向量长度
        n_head: 多头注意力的头数
        n_embd // n_head: 每个头的维度

        ln1, ln2: Layer Normalization 层归一化
        sa : Self-Attention 自注意力
        ffwd : Feed-Forward Network(有时也写FFN)前馈网络
        '''
        super().__init__()

        # 1. 层归一化
        self.ln1 = nn.LayerNorm(n_embd)
        # 2. 多头注意力(命名为雷达sa)， 通过Q/K/V, 根据与其他词的相关性重新计算自己的表达
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        # 3. 前馈网络(消化)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

        # 多头注意力之前和前馈网络前都有一个LayerNorm, 
    
    def forward(self, x):
        # 残差连接： output = x + f(x)
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x
    
class NanoGPT(nn.Module):
    def forward(self, idx, targets=None):
        '''
        idx: 输入字符的ID序列
        target: 
        '''

        # 1. 加上位置编码，让模型了解顺序
        x = self.token_embedding_table(idx) + self.position_embedding_table

        # 2. 经过多层Transformer 块
        x = self.blocks(x)

        # 3. 归一化并输出概率
        logits = self.lm_head(self.ln1(x))

        # LM Head (Language Model Head): 这是一个线性层，它的作用是把隐藏层的向量投影到词表大小的空间。
        # Logits: 它的输出是一个分数值（Logits）。比如你的词表有 5000 个字，它就会输出 5000 个数。哪个数字最大，就代表模型认为下一个字最可能是谁。

        return logits

2. 组装之前：

训练大模型时，需遵循一个标准化循环：

“读取-计算-对答案-改错”.



In [ ]:
# 1. 数据加载：
def get_batch(split):
    # 根据是训练集还是验证集，取不同的数据
    data = train_data if split == 'train' else val_data
    
    # 随机产生 batch_size 个起始位置
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x 是输入，y 是对应的标签（即 x 后面那个字）
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    # 搬运到 GPU（如果有的话）
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
# 2. 核心训练循环，几乎所有深度学习模型的训练都是这样：

# 初始化优化器（就像是一个负责调螺丝的机械师）
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for iter in range(max_iters):
    # 1. 采样一波数据
    xb, yb = get_batch('train')

    # 2. 前向传播（让模型做题）
    logits, loss = model(xb, yb)

    # 3. 反向传播之前，先清空之前的“纠错记录”
    optimizer.zero_grad(set_to_none=True)

    # 4. 计算梯度（算出哪些参数该往哪边调）
    loss.backward()

    # 5. 更新参数（动手调螺丝）
    optimizer.step()

    # 定期打印进度
    if iter % eval_interval == 0:
        print(f"步数 {iter}: 损失值 (Loss) 为 {loss.item():.4f}")